# 🛒 Notebook 2: Worked example — Order Service

We'll build a tiny **Order Service** that must call two downstream microservices on every request:

1. **Inventory Service** — reserve the items in stock. High traffic, can occasionally get slow.
2. **Shipping Service** — schedule the shipment. Lower traffic, usually reliable.

Today, the **Inventory Service is having a bad day** and its latency spiked from 50 ms to 2 s.
Let's see what happens with and without bulkheads.

> This example mirrors the scenario from `grokking-microservices-design-patterns` (see `../references/designgurus.md`).


## 🛠️ Setup

```bash
cd 05-microservices/bulkhead
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear: `Cmd+Shift+P` → **Reload Window**.


## 🎭 The two fake downstreams

These are stand-ins for real HTTP/gRPC services — `time.sleep` fakes the network latency.

In [1]:
import time
from concurrent.futures import ThreadPoolExecutor, TimeoutError

# Inventory is slow today (2 s). In a healthy world this would be ~50 ms.
def inventory_service(order_id: int) -> str:
    time.sleep(2.0)
    return f"inventory-reserved-for-{order_id}"

# Shipping is fast and healthy.
def shipping_service(order_id: int) -> str:
    time.sleep(0.05)
    return f"shipping-scheduled-for-{order_id}"


## ❌ Bad: one shared thread pool

The service has TWO kinds of incoming traffic:

- `POST /orders` — calls slow inventory + shipping.
- `GET /shipping-status` — *only* calls shipping (fast, should respond in ~50 ms).

With a single shared pool, the slow inventory calls fill the pool and `/shipping-status` is starved too, even though shipping itself is perfectly healthy.

In [2]:
shared_pool = ThreadPoolExecutor(max_workers=4, thread_name_prefix="shared")

def create_order_BAD(order_id: int) -> dict:
    f_inv  = shared_pool.submit(inventory_service, order_id)
    f_ship = shared_pool.submit(shipping_service, order_id)
    return {"inventory": f_inv.result(), "shipping": f_ship.result()}

def shipping_status_BAD() -> str:
    # Goes through the SAME shared pool
    return shared_pool.submit(shipping_service, -1).result()

# 1) Launch a burst of slow orders that saturate the shared pool.
#    We use one outer pool (not 4 throwaway executors) so nothing leaks.
caller_pool = ThreadPoolExecutor(max_workers=4, thread_name_prefix="http")
burst = [caller_pool.submit(create_order_BAD, i) for i in range(4)]
time.sleep(0.1)  # let the burst grab shared_pool slots

# 2) Now measure a healthy shipping-status call
t0 = time.time()
shipping_status_BAD()
shipping_only_latency_bad = time.time() - t0
print(f"/shipping-status latency while inventory is overloaded: {shipping_only_latency_bad:.2f}s")
print("-> shipping is healthy but looks broken because of inventory")

for f in burst: f.result()  # drain
caller_pool.shutdown(wait=True)
shared_pool.shutdown(wait=True)


/shipping-status latency while inventory is overloaded: 2.07s
-> shipping is healthy but looks broken because of inventory


A shipping-only request that should take ~50 ms took seconds. Every endpoint in your service now appears broken, even the ones that don't touch the sick dependency. This is **failure propagation**.

## ✅ Good: separate bulkheads per downstream

Give each downstream its own bounded pool. The slow inventory calls can't starve shipping, because shipping has its own dedicated threads.

In [3]:
# Size each bulkhead based on the downstream's safe concurrency.
inventory_pool = ThreadPoolExecutor(max_workers=4, thread_name_prefix="inv")
shipping_pool  = ThreadPoolExecutor(max_workers=4, thread_name_prefix="ship")

def create_order_GOOD(order_id: int) -> dict:
    f_inv  = inventory_pool.submit(inventory_service, order_id)
    f_ship = shipping_pool.submit(shipping_service, order_id)
    return {"inventory": f_inv.result(), "shipping": f_ship.result()}

def shipping_status_GOOD() -> str:
    # Uses the shipping bulkhead - completely separate from inventory
    return shipping_pool.submit(shipping_service, -1).result()

# 1) Same burst of slow orders - now they can only saturate the INVENTORY bulkhead
caller_pool = ThreadPoolExecutor(max_workers=4, thread_name_prefix="http")
burst = [caller_pool.submit(create_order_GOOD, i) for i in range(4)]
time.sleep(0.1)

# 2) Measure the healthy shipping-status call
t0 = time.time()
shipping_status_GOOD()
shipping_only_latency_good = time.time() - t0
print(f"/shipping-status latency while inventory is overloaded: {shipping_only_latency_good:.2f}s")
print("-> shipping stays fast because it has its own bulkhead")

for f in burst: f.result()
caller_pool.shutdown(wait=True)


/shipping-status latency while inventory is overloaded: 0.19s
-> shipping stays fast because it has its own bulkhead


Same failure in inventory, totally different user experience:

- **Bad**: every endpoint slow.
- **Good**: endpoints that don't depend on inventory stay fast.

We haven't magically fixed inventory — callers of `/orders` still wait. But **the blast radius is now contained** to the one endpoint that actually needs inventory.

## ✨ Best: bulkhead + timeout + graceful fallback

Bulkheads alone don't rescue **callers of the slow dependency**. If inventory stays slow for minutes, clients still wait 2 s per order. Combine three patterns:

1. **Bulkhead** — contain the failure.
2. **Timeout** — don't wait forever for a sick dependency.
3. **Fallback** — return a degraded-but-useful response when the dependency fails.

In [4]:
INVENTORY_TIMEOUT_S = 0.5   # tune to 99th-percentile *healthy* latency + headroom
SHIPPING_TIMEOUT_S  = 0.3

def create_order_BEST(order_id: int) -> dict:
    f_inv  = inventory_pool.submit(inventory_service, order_id)
    f_ship = shipping_pool.submit(shipping_service, order_id)

    # Inventory is critical - can't sell what we can't reserve.
    try:
        inv_result = f_inv.result(timeout=INVENTORY_TIMEOUT_S)
    except TimeoutError:
        # Note: cancel() only cancels futures that haven't started yet.
        # Running ones will keep occupying a bulkhead slot until they finish -
        # that's why bounding the pool matters so much.
        f_inv.cancel()
        # Fallback: queue the reservation asynchronously and warn the user
        inv_result = "INVENTORY_PENDING"

    # Shipping quote is nice-to-have - degrade gracefully.
    try:
        ship_result = f_ship.result(timeout=SHIPPING_TIMEOUT_S)
    except TimeoutError:
        f_ship.cancel()
        ship_result = "SHIPPING_ESTIMATE_UNAVAILABLE"

    return {"order_id": order_id, "inventory": inv_result, "shipping": ship_result}

# Show that each order now returns fast, with a degraded inventory field.
t0 = time.time()
order_pool = ThreadPoolExecutor(max_workers=10, thread_name_prefix="http")
results = list(order_pool.map(create_order_BEST, range(5)))
elapsed = time.time() - t0

for r in results:
    print(r)
print(f"\nhandled {len(results)} orders in {elapsed:.2f}s - fast responses, degraded where needed")

order_pool.shutdown(wait=True)
inventory_pool.shutdown(wait=False)
shipping_pool.shutdown(wait=False)


{'order_id': 0, 'inventory': 'INVENTORY_PENDING', 'shipping': 'shipping-scheduled-for-0'}
{'order_id': 1, 'inventory': 'INVENTORY_PENDING', 'shipping': 'shipping-scheduled-for-1'}
{'order_id': 2, 'inventory': 'INVENTORY_PENDING', 'shipping': 'shipping-scheduled-for-2'}
{'order_id': 3, 'inventory': 'INVENTORY_PENDING', 'shipping': 'shipping-scheduled-for-3'}
{'order_id': 4, 'inventory': 'INVENTORY_PENDING', 'shipping': 'shipping-scheduled-for-4'}

handled 5 orders in 0.65s - fast responses, degraded where needed


### What the user actually sees

- Shipping estimate arrives in ~50 ms as usual.
- Inventory reservation shows `INVENTORY_PENDING` — the UI can say *"we're reserving your items, we'll email you in a minute."*
- The service stays responsive **even while an entire downstream is struggling**.

That's the whole point of resilience patterns: **graceful degradation instead of catastrophic failure.**

## 📏 How to size a bulkhead

Pool size is the most-asked question. A practical starting formula:

```
pool_size ≈ target_RPS × p99_latency_seconds × safety_factor
```

- **target_RPS** — how many calls per second you expect to send.
- **p99_latency_seconds** — typical 99th-percentile response time of the downstream (when healthy).
- **safety_factor** — 1.2 to 2× to absorb bursts.

Example: 50 RPS × 0.1 s × 1.5 = **~8 threads**.

Then:
1. **Load test** to validate.
2. Add dashboards for pool **saturation** (`active / max`) and **rejections**.
3. Tune so rejections happen *before* the downstream itself gets overloaded.

## 🔎 Observability: you can't tune what you can't see

A bulkhead is only useful if you know when it's near full. The two metrics every team should export (Prometheus, Datadog, CloudWatch, etc.) are:

| Metric | What it tells you | Alert when |
|---|---|---|
| **saturation** = `active / max` per pool | How close to exhausted the bulkhead is | > 80% sustained |
| **rejections / sec** per pool | Callers turned away (semaphore bulkheads) | > 0 sustained |

Bonus metrics: per-pool p50/p99 latency, queue depth (for thread pools), and time-in-queue.

A good dashboard answers: *"which downstream is about to drown?"* before your users notice.

## 🌍 Real-world tools that give you bulkheads

| Tool / Platform | How it provides bulkheads |
|---|---|
| **Netflix Hystrix** (historical) | Thread-pool or semaphore isolation per command |
| **Resilience4j** (Java) | `Bulkhead` and `ThreadPoolBulkhead` modules |
| **Polly** (.NET) | `BulkheadPolicy` |
| **Envoy / Istio** | Per-upstream `max_connections`, `max_pending_requests`, `max_requests` |
| **AWS** | SQS queues per workload, separate Lambda concurrency limits per function |
| **Kubernetes** | Separate Deployments/namespaces, `resources.limits`, HPA per workload |
| **Databases** | Per-service connection pools (PgBouncer pools, HikariCP pools) |

## 🧭 When *not* to use a bulkhead

- You only have one downstream. (You still want a bounded pool, but it isn't partitioning anything.)
- The overhead of extra threads/connections exceeds the risk being mitigated.
- Fairness matters more than isolation — e.g., a single FIFO queue may be preferred for strict ordering.

## 🧠 Exercises

1. Change `INVENTORY_TIMEOUT_S` to `2.5` in the *Best* cell. What happens? Why is that worse than the smaller timeout?
2. Add a third downstream — `recommendations_service` — with its own pool. Try making it the slow one and see that inventory and shipping stay snappy.
3. Replace the `ThreadPoolExecutor`-based `inventory_pool` with the `SemaphoreBulkhead` from notebook 1. When would semaphore-based isolation be the better choice? (Hint: async I/O.)

## 🎯 Takeaways

- **Bulkheads contain failures**, but don't remove them.
- **Every downstream deserves its own bounded pool** — thread pool, connection pool, or semaphore.
- **Combine with timeouts + fallbacks** for full graceful degradation.
- **Size pools from data**, not intuition. Then alert on saturation.
- Bulkheads are one layer in a stack that also includes **retries**, **circuit breakers**, **rate limiting**, and **load shedding**.
